env --> yolo

In [2]:
import os
import random
import shutil
import glob

## Split data 

### dataset split =>
### s1 = 513, 103, 68 (75%, 15%,10%)
### s2 =  385, 103, 68 (train_s2 = 75% of train_s1)
### s3 =  256, 103, 68 (train_s2 = 50% of train_s1)

In [ ]:
TRAIN_RATIO = 0.75
VAL_RATIO = 0.15
TEST_RATIO = 0.10

## set seed (458)
SEED = 458 
random.seed(SEED)

In [ ]:
DATASET_DIR = "/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280/ALL"          # your input folder
OUTPUT_DIR = "/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280_newSplit"    # output folder


images = []

images = glob.glob(os.path.join(DATASET_DIR, "*.jpg"))
print(f"Total images: {len(images)}")

# keep only images that have matching txt
pairs = []
for img_path in images:
    txt_path = img_path.replace(".jpg", ".txt")
    if os.path.exists(txt_path):
        pairs.append((img_path, txt_path))

print(f"Found {len(pairs)} valid image-txt pairs.")
print(f"Sample pair 52: {pairs[52]}")

# shuffle and then split
random.shuffle(pairs)
print(f"Sample pair 52: {pairs[52]}")

total = len(pairs)
train_end = (round(total * TRAIN_RATIO))
val_end = train_end + (round(total * VAL_RATIO))

train_data_s1 = pairs[:train_end]
val_data = pairs[train_end:val_end]
test_data = pairs[val_end:]

print(f"Train: {len(train_data_s1)}")
print(f"Val: {len(val_data)}")
print(f"Test: {len(test_data)}")

## make dirs
for split in ["train_s1", "val", "test"]:
    for subdir in ["images", "labels"]:
        os.makedirs(os.path.join(OUTPUT_DIR, split, subdir), exist_ok=True)

## copy data
for split, data in [("train_s1", train_data_s1), ("val", val_data), ("test", test_data)]:
    for img, txt in data:
        shutil.copy(img, os.path.join(OUTPUT_DIR, split, "images", os.path.basename(img)))
        shutil.copy(txt, os.path.join(OUTPUT_DIR, split, "labels", os.path.basename(txt)))

################################################################################

# train_s2: 75% of train_s1, which is 385 of original train
## shuffle and then split
print(f"Sample pair 13 before shuffle: {train_data_s1[13]}")
random.shuffle(train_data_s1)
print(f"Sample pair 13 after shuffle: {train_data_s1[13]}")


TRAIN_RATIO_s2 = 0.75
total = len(train_data_s1) ## total = 75% of train_s1, which is 385 of original train
train_end = (round(total * TRAIN_RATIO_s2))

## subsample
train_data_s2 = train_data_s1[:train_end]
print(f"Length of train_data_s2: {len(train_data_s2)}")

# make dirs
for subdir in ["images", "labels"]:
    os.makedirs(os.path.join(OUTPUT_DIR, "train_s2", subdir), exist_ok=True)

# copy
for img, txt in train_data_s2:
    shutil.copy(img, os.path.join(OUTPUT_DIR, "train_s2", "images", os.path.basename(img)))
    shutil.copy(txt, os.path.join(OUTPUT_DIR, "train_s2", "labels", os.path.basename(txt)))

################################################################################

# train_s3: 50% of train_s1, which is 257 of original train
## shuffle and then split
print(f"Sample pair 13 before shuffle: {train_data_s2[13]}")
random.shuffle(train_data_s2)
print(f"Sample pair 13 after shuffle: {train_data_s2[13]}")


TRAIN_RATIO_s3 = 0.50
total = len(train_data_s1)
train_end = (round(total * TRAIN_RATIO_s3))

## subsample
train_data_s3 = train_data_s2[:train_end]
print(f"Length of train_data_s3: {len(train_data_s3)}")

# make dirs
for subdir in ["images", "labels"]:
    os.makedirs(os.path.join(OUTPUT_DIR, "train_s3", subdir), exist_ok=True)

# copy 
for img, txt in train_data_s3:
    shutil.copy(img, os.path.join(OUTPUT_DIR, "train_s3", "images", os.path.basename(img)))
    shutil.copy(txt, os.path.join(OUTPUT_DIR, "train_s3", "labels", os.path.basename(txt)))

Total images: 684
Found 684 valid image-txt pairs.
Sample pair 52: ('/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280/ALL/Carabidae_678.jpg', '/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280/ALL/Carabidae_678.txt')
Sample pair 52: ('/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280/ALL/Carabidae_380.jpg', '/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280/ALL/Carabidae_380.txt')
Train: 513
Val: 103
Test: 68
Sample pair 13 before shuffle: ('/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280/ALL/Carabidae_501.jpg', '/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280/ALL/Carabidae_501.txt')
Sample pair 13 after shuffle: ('/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280/ALL/Carabidae_80.jpg', '/clu

## K-fold test split S1

### k = 4

In [47]:
k = 4

## set seed (5829)
SEED = 5829 
random.seed(SEED)

In [48]:
k_fold_dir = "/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280_newSplit/k-fold_sets_s1"
os.makedirs(k_fold_dir, exist_ok=True)

#  make dirs
for subdir in ["set1", "set2", "set3", "set4"]:
    os.makedirs(os.path.join(k_fold_dir, subdir), exist_ok=True)
    for split in ["images", "labels"]:
        os.makedirs(os.path.join(k_fold_dir, subdir, split), exist_ok=True)
    

In [ ]:
train_DIR = "/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280_newSplit/train_s1"          # input folder
val_DIR = "/cluster/home/sbhansali/digi_hiwi/drawers2/datasets/all_jpg_qr_scale_whitebal_crop_pad_1280_newSplit/val"    # output folder

train_pairs = []
for img_path in glob.glob(os.path.join(train_DIR, "images/*.jpg")):
    txt_path = img_path.replace(".jpg", ".txt")
    txt_path = txt_path.replace("/images/", "/labels/")
    if os.path.exists(txt_path):
        train_pairs.append((img_path, txt_path))


val_pairs = []
for img_path in glob.glob(os.path.join(val_DIR, "images/*.jpg")):
    txt_path = img_path.replace(".jpg", ".txt")
    txt_path = txt_path.replace("/images/", "/labels/")
    if os.path.exists(txt_path):
        val_pairs.append((img_path, txt_path))


all_pairs = train_pairs + val_pairs
print(len(all_pairs))

# shuffle and then split into 4
random.shuffle(pairs)



616


In [ ]:
# Number of files per set
set_size = len(all_pairs) // k

# Create 4 sets
set1 = all_pairs[0:(set_size)]
set2 = all_pairs[set_size:set_size*2]
set3 = all_pairs[set_size*2:set_size*3]
set4 = all_pairs[set_size*3:]

all_sets = [set1, set2, set3, set4]

## copy data
for set_idx, s in enumerate(all_sets):
    split = "set" + str(set_idx + 1)
    for i in s:
        img = i[0]
        txt = i[1]
        shutil.copy(img,os.path.join(k_fold_dir, split, "images", os.path.basename(img)))
        shutil.copy(txt,os.path.join(k_fold_dir, split, "labels", os.path.basename(txt)))